<a href="https://colab.research.google.com/github/marcoskantor/Curso_Introduccion_Ingenieria_Datos/blob/main/Proyecto_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto Final — Entregable Técnico
## Calidad de los datos públicos de radiografías lumbares para un sistema de soporte a la decisión clínica (CDSS) de inestabilidad espinal

**Tesista:** Marcos Kantor · **Director:** Ramiro M. Irastorza (IFLySiB-CONICET) · **Codirector:** Marcelo A. Cappelletti (LEICI-CONICET, UNAJ) · **Materia:** (20262Q) DIA.04 — Introducción a la Ingeniería de Datos — Comisión A

La tesis busca un CDSS que asista el cálculo del *Spinal Instability Neoplastic Score* (SINS), que se evalúa vértebra por vértebra: su primera etapa es **localizar cada vértebra** en la radiografía.

> **Pregunta de investigación: ¿los datos públicos de radiografías lumbares tienen la cobertura, el balance, la calidad geométrica y la coherencia anatómica necesarias para entrenar el detector de vértebras L1–L5 del CDSS?**

| | Pregunta de análisis |
|---|---|
| P1 | Cobertura y balance: ¿cuántas imágenes tienen L1–L5 completas? ¿Alcanza sumando una segunda fuente? |
| P2 | Calidad geométrica: ¿qué anotaciones tienen un tamaño o una forma atípicos para su vértebra? |
| P3 | Coherencia anatómica: ¿se respeta el orden L1→L5? ¿Cómo están orientadas las radiografías? |
| P4 | Representatividad: ¿qué diagnósticos, sexos y edades cubre? |
| P5 | Alineación: ¿la inclinación medida con momentos de imagen (OpenCV) distingue las patologías? |

**Organización.** Este cuaderno contiene el proceso completo, sus justificaciones y el código que demuestra cada requisito; las funciones auxiliares (descarga, parsing, descriptores, carga de las bases) están en `src/` y el DDL en `sql/` del mismo repositorio público, que la primera celda de código clona. El análisis exploratorio y todas las figuras están en el anexo [`notebooks/Proyecto_final_extendido.ipynb`](notebooks/Proyecto_final_extendido.ipynb).

## Declaración de uso de IA

- "Claude Opus 5.5" para "generar y revisar el código Python del cuaderno y de los módulos de `src/` (extracción con verificación SHA-256, parsing de anotaciones, control de calidad, descriptores con momentos de OpenCV, bases SQLite transaccional y almacén, tablero Streamlit), redactar el borrador de las celdas Markdown y de la presentación de la defensa, y revisar los gráficos con criterios de *storytelling*".

Se usó a través de Claude Code (Anthropic); no se usó ninguna otra herramienta de IA. Las decisiones metodológicas (criterios de descarte e imputación, umbrales de atípicos, grano de las tablas de hechos, mensaje del tablero) fueron definidas y validadas por el autor.

## Instrucciones de la IA (*Human-in-the-loop*)

Instrucciones para reproducir el resultado con la herramienta de IA, cada una seguida de la verificación humana que hice antes de avanzar.

1. **Extracción reproducible.** *"Sin credenciales ni archivos subidos a mano: descargá por URL pública los zips crudos (CC BY 4.0) que publiqué en `data/raw/` de mi repositorio, verificá su SHA-256 y descomprimilos en un área de staging, de forma idempotente."* *Verificación humana:* el hash coincide y el cuaderno corre con *Ejecutar todo* en una sesión nueva de Colab.
2. **Inventario, metadatos y parsing.** *"Inventariá cada imagen (partición, resolución, formato de píxel, si tiene etiquetas). Extraé diagnóstico, sexo y edad del nombre de archivo con expresiones regulares sin inventar valores (en `L4-5`, el 5 no es una edad). Parseá las etiquetas YOLOv8 de segmentación, desnormalizá a píxeles, calculá el área por la fórmula de Gauss y guardá anotaciones y vértices en staging."* *Verificación humana:* revisé la tabla nombre → metadatos y superpuse los polígonos sobre una radiografía.
3. **Carga optimizada.** *"Leé el staging eligiendo columnas (`usecols`) y tipando cada variable (`int8`, `int16`, `float32`, `category`, `Int8` nullable para la edad); reportá el ahorro de memoria."*
4. **Faltantes y duplicados.** *"Analizá los faltantes en tres niveles (imagen sin etiquetas, vértebra ausente, atributo inválido) y en los metadatos. No imputes geometría ni datos clínicos: marcá y conservá. Si una vértebra aparece dos veces en una imagen, conservá la más cercana a la posición mediana de esa vértebra."* *Verificación humana:* revisé el duplicado a mano.
5. **Atípicos.** *"Descartá sólo lo geométricamente inválido. Sobre variables normalizadas dentro de cada imagen, probá normalidad con Shapiro-Wilk para justificar Tukey 2·RIC en lugar de Z-score, sumá una distancia de Mahalanobis robusta (mediana y MAD) y un control de coherencia anatómica L1→L5; marcá cada atípico con su motivo, no lo borres."* *Verificación humana:* rechacé el área relativa a la imagen (49 falsos atípicos por el encuadre) y pasé a normalizar contra las vecinas.
6. **Momentos de OpenCV (P5).** *"Con `cv2.moments` calculá área (validada contra Gauss), centroide e inclinación θ = ½·atan2(2μ11, μ20 − μ02); armá el grano segmento intervertebral y el Cobb aproximado, y probá con Kruskal-Wallis y Mann-Whitney si distinguen las patologías, según la vista de cada radiografía."* *Verificación humana:* m00 coincide con el área y no sobreinterpreté resultados con n ≈ 8.
7. **Segunda fuente y orientación.** *"Aplicá a NHANES el mismo control adaptado a cajas. Si el control de orden marca muchas cajas, revisá si son errores de etiqueta o radiografías giradas, y separá orientación de orden."* *Verificación humana:* revisé las 1.709 marcas iniciales: eran imágenes giradas o invertidas.
8. **Base transaccional y almacén.** *"Diseñá un SQLite al menos en 2FN (vértices en su propia tabla por 1FN; anotación con clave compuesta (imagen, vértebra) sin dependencias parciales) y un copo de nieve con dos tablas de hechos que compartan dimensiones (constelación): grano 'vértebra en una imagen' y 'segmento en una imagen', PK compuestas por las claves de sus dimensiones, medidas atómicas y sólo las que responden P1–P5."* *Verificación humana:* contrasté cada métrica con P1–P5 y validé la integridad referencial.
9. **Tablero y comunicación.** *"Construí un tablero Streamlit que lea sólo del almacén, con un veredicto ajustable y la evidencia de P1–P5, siguiendo Storytelling with Data: el título enuncia la conclusión, gris de contexto, un color de acento y ejes de conteo enteros."*
10. **Extensión del cuaderno.** *"El cuaderno no puede superar 15 páginas: dejá la narrativa, las justificaciones y el código que demuestra cada requisito (carga optimizada, faltantes, atípicos, consultas al almacén) y mové las funciones auxiliares a `src/` y el DDL a `sql/`, sin cambiar ningún resultado."* *Verificación humana:* comparé tabla por tabla el almacén generado con el de la versión extendida: son idénticos.

## 1. Datos y extracción

| | Lumbar Spine (principal) | NHANES L1–L5 (segunda fuente) |
|---|---|---|
| Origen | Roboflow Universe, `rp-team/lumbar-spine-vpxwf` v1 | Roboflow Universe, `main-kybbe/roboflow_nhanes_l1_to_l5_detection` v1 |
| Contenido | 50 radiografías de frente y perfil, polígonos L1–L5; diagnóstico, sexo y edad en el nombre de archivo | 4.436 radiografías de perfil (encuesta NHANES, EE. UU.), cajas L1–L5 estiradas a 640×640 |
| Responde | P1 a P5 | P1 y P3 (una caja no tiene forma ni inclinación) |

Las dos tienen licencia **CC BY 4.0**. Roboflow exige una API key, así que las copias crudas sin modificar están en `data/raw/` del repositorio público: se descargan por URL y se verifica su **SHA-256** (cualquiera obtiene los mismos bytes). De NHANES sólo se redistribuyen las anotaciones COCO, porque P1 y P3 no necesitan los píxeles.

In [ ]:
!pip -q install streamlit plotly
import os, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

# Funciones auxiliares (src/) y DDL (sql/) del repositorio público. Fuera de Colab, PF_BASE puede apuntar a una copia local.
REPO, RAMA = "https://github.com/marcoskantor/Curso_Introduccion_Ingenieria_Datos", "proyecto-final-15-paginas"
EN_COLAB = "google.colab" in sys.modules
BASE = Path(os.environ.get("PF_BASE", "/content/repo" if EN_COLAB else "repo"))
if not BASE.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", RAMA, REPO + ".git", str(BASE)], check=True)
sys.path.insert(0, str(BASE / "src"))
import extraccion as ex, calidad as cal, forma, almacen, graficos

graficos.estilo(); pd.set_option("display.width", 160)
WORK = Path("/content/proyecto_final") if EN_COLAB else Path("proyecto_final")
RAW, RAW2, STG = WORK / "raw", WORK / "raw_nhanes", WORK / "staging"
STG.mkdir(parents=True, exist_ok=True)
OLTP_DB, DW_DB = WORK / "spine_oltp.db", WORK / "spine_dw.db"
DATOS = REPO + "/raw/main/data/raw/"
DATA_SHA256 = "b62d0ce506aad57f93ec9f9e8c5cadb52173d25689b76ae1402109c7768b3096"
NHANES_SHA256 = "149896dcd3359e9942df929771a89cfd2a3809c7ab90ecb451628e54334ebb71"
UMBRAL_TUKEY, ALPHA = 2.0, 0.999          # k de Tukey y confianza del límite χ² de Mahalanobis

In [ ]:
ex.descomprimir(ex.descargar(DATOS + "lumbar-spine-vpxwf-v1-yolov8.zip", DATA_SHA256, WORK), RAW)
ex.descomprimir(ex.descargar(DATOS + "nhanes-l1-l5-v1-coco-anotaciones.zip", NHANES_SHA256, WORK), RAW2)
CLASES = ex.clases(RAW)
inventario = ex.inventariar(RAW)                   # una fila por imagen, con metadatos extraídos por regex
rechazos = ex.parsear_yolo(inventario, RAW, STG)   # escribe el staging: imagenes, anotaciones y vertices (CSV)
img2, anot2 = ex.leer_nhanes(RAW2)
print(f"Lumbar Spine: {len(inventario)} imágenes, {(inventario.n_lineas > 0).sum()} con anotaciones, "
      f"{len(rechazos)} líneas rechazadas | clases {CLASES} | NHANES: {len(img2):,} imágenes, {len(anot2):,} cajas")

## 2. Carga optimizada

El staging se lee **eligiendo columnas** (`usecols`: quedan afuera `n_lineas` y `gris_en_rgb`, que sólo sirvieron para el diagnóstico) y **tipando** cada variable según su dominio: `int8` para clase, línea, canales y bits (valores chicos); `int16` para vértices y resolución (< 32.767 px); `Int8` *nullable* para la edad (< 128 y puede faltar, cosa que `int8` no admite); `float32` para coordenadas y áreas (precisión sub-píxel sobrada) y `category` para texto muy repetido.

In [ ]:
TIPOS = {
    "imagenes":    dict(usecols=["archivo", "split", "ancho_px", "alto_px", "tiene_etiquetas", "diagnostico", "sexo",
                                 "edad", "nivel_listesis", "formato", "canales", "profundidad_bits", "tamano_kb"],
                        dtype={"archivo": "category", "split": "category", "ancho_px": "int16", "alto_px": "int16",
                               "tiene_etiquetas": "bool", "diagnostico": "category", "sexo": "category", "edad": "Int8",
                               "nivel_listesis": "category", "formato": "category", "canales": "int8",
                               "profundidad_bits": "int8", "tamano_kb": "float32"}),
    "anotaciones": dict(usecols=["archivo", "linea", "clase_id", "n_vertices", "x_min", "y_min", "x_max", "y_max",
                                 "area_px", "cx", "cy"],
                        dtype={"archivo": "category", "linea": "int8", "clase_id": "int8", "n_vertices": "int16",
                               **{c: "float32" for c in ["x_min", "y_min", "x_max", "y_max", "area_px", "cx", "cy"]}}),
    "vertices":    dict(usecols=["archivo", "linea", "orden", "x_px", "y_px"],
                        dtype={"archivo": "category", "linea": "int8", "orden": "int16", "x_px": "float32", "y_px": "float32"}),
}
memoria = lambda df: df.memory_usage(deep=True).sum() / 1024
tablas, ahorro = {}, []
for nombre, kw in TIPOS.items():
    crudo = pd.read_csv(STG / f"{nombre}.csv")
    tablas[nombre] = pd.read_csv(STG / f"{nombre}.csv", **kw)
    ahorro.append((nombre, len(crudo), memoria(crudo), memoria(tablas[nombre])))
imagenes, anotaciones, vertices = tablas["imagenes"], tablas["anotaciones"], tablas["vertices"]
ahorro = pd.DataFrame(ahorro, columns=["tabla", "filas", "KiB sin optimizar", "KiB optimizado"]).set_index("tabla")
ahorro["ahorro %"] = 100 * (1 - ahorro["KiB optimizado"] / ahorro["KiB sin optimizar"])
ahorro.round(1)

## 3. Calidad de datos
### 3.1 Valores faltantes y duplicados

| Nivel | Qué falta | Decisión y justificación |
|---|---|---|
| Imagen | archivo de etiquetas ausente o vacío | **se conserva** con `anotada = 0`: la geometría de una vértebra no se puede imputar y descartar la imagen ocultaría el problema de cobertura (P1) |
| Vértebra | una imagen anotada no tiene alguna de L1–L5 | **se conserva** y se mide la **completitud**: el faltante es informativo (P1) |
| Atributo | coordenadas nulas o clase fuera de `data.yaml` | **descarte** de la anotación: sin coordenadas o sin clase no hay dato utilizable |
| Metadato | sexo, edad o diagnóstico ilegible en el nombre de archivo | **queda nulo** ("sin dato" en el tablero): imputarlo inventaría pacientes y sesgaría P4 |

**Duplicados.** El grano es una anotación por vértebra y por imagen. Si una vértebra aparece dos veces se conserva la de posición vertical relativa más cercana a la mediana de esa vértebra (la elección más consistente con la anatomía) y el descarte queda registrado.

In [ ]:
imagenes["anotada"] = imagenes["archivo"].isin(anotaciones["archivo"]).astype("int8")
clase_invalida = ~anotaciones["clase_id"].between(0, len(CLASES) - 1)
nulos = int(anotaciones.isna().sum().sum())
bitacora = [("Anotaciones parseadas", len(anotaciones))]
antes = len(anotaciones)
anotaciones = anotaciones[~clase_invalida].dropna().copy()
bitacora.append(("Descartadas por nulos o clase inválida", antes - len(anotaciones)))
anotaciones["vertebra"] = pd.Categorical(anotaciones["clase_id"].map(dict(enumerate(CLASES))), categories=CLASES)
completitud = (anotaciones.groupby("archivo", observed=True)["vertebra"].nunique()
               .reindex(imagenes.loc[imagenes.anotada == 1, "archivo"]).fillna(0).astype(int))
sin_anotar = imagenes.loc[imagenes.anotada == 0, "diagnostico"].astype(str).value_counts().to_dict()
print(f"Imágenes sin anotaciones: {len(imagenes) - imagenes.anotada.sum()} de {len(imagenes)}, por diagnóstico {sin_anotar}")
print(f"Nulos en anotaciones: {nulos} | clase inválida: {int(clase_invalida.sum())} | "
      f"metadatos sin dato: {imagenes[['diagnostico', 'sexo', 'edad']].isna().sum().to_dict()}")
print("Imágenes anotadas según cuántas vértebras distintas tienen:", completitud.value_counts().sort_index().to_dict())

anotaciones = anotaciones.merge(imagenes[["archivo", "ancho_px", "alto_px"]], on="archivo", how="left")
anotaciones["cy_rel"] = anotaciones["cy"] / anotaciones["alto_px"]       # posición vertical relativa (0 = arriba)
anotaciones, dup, n_dup = cal.deduplicar(anotaciones, ["archivo", "vertebra"])
bitacora.append(("Descartadas por duplicado (imagen, vértebra)", n_dup))
dup[["vertebra", "linea", "cy_rel", "dist_mediana", "area_px"]].round(3)

**Lectura.** El 32 % de las imágenes de Lumbar Spine no tiene anotaciones y ese faltante **no es aleatorio**: son todas de escoliosis o del código «S». Entre las anotadas, 33 de 34 tienen las cinco vértebras. Hay un duplicado (dos L2 en una imagen): se conserva la de la posición esperada.

### 3.2 Datos atípicos

* **Inválidos geométricos** (área < 1 px² o contorno que sale de la imagen más de un 2 %): se **descartan**, no representan una vértebra; los desbordes menores son redondeo del anotador y se recortan al borde.
* **Atípicos estadísticos por vértebra, sobre variables sin escala.** Las imágenes mezclan recortes lumbares y columnas completas (de 144×606 a 1669×3246 px), así que el área en píxeles o relativa a la imagen depende del encuadre (marcaba 49 falsos atípicos). Se normaliza **dentro de cada imagen**: `area_norm` = área / mediana de sus vecinas, `aspecto` = ancho / alto de la caja y `relleno` = área del polígono / área de la caja. Como **Shapiro-Wilk** rechaza la normalidad en la mayoría de los casos, el corte |Z| > 3 no tiene sustento: se usan **vallas de Tukey** (Q1 − 2·RIC, Q3 + 2·RIC, más conservadoras que 1,5) y una **distancia de Mahalanobis robusta** (centro = mediana, escala = MAD, límite χ² al 99,9 % con 3 g.l.) que detecta combinaciones raras.
* **Incoherencias anatómicas:** el centroide vertical debe crecer de L1 a L5.

**Decisión.** Los atípicos estadísticos y anatómicos se **marcan** con su motivo (`es_atipico`, `orden_coherente`) y **no se descartan**: con 169 anotaciones pueden ser variación anatómica real, y excluirlos queda disponible como filtro en el almacén y el tablero.

In [ ]:
anotaciones, n_fuera, n_degenerados = cal.invalidos_geometricos(anotaciones)
bitacora.append(("Descartadas por geometría inválida", n_fuera + n_degenerados))
anotaciones = cal.variables_relativas(anotaciones)
VARS = cal.VARS                                       # area_norm, aspecto, relleno
sw = cal.normalidad(anotaciones)
print(f"Shapiro-Wilk rechaza la normalidad (α = 0,05) en {(sw.p_shapiro < 0.05).sum()} de {len(sw)} vértebra × variable")

motivos = {i: [] for i in anotaciones.index}           # una anotación puede acumular varios motivos
for v, g in anotaciones.groupby("vertebra", observed=True):            # vallas de Tukey dentro de cada vértebra
    for var in VARS:
        q1, q3 = g[var].quantile([0.25, 0.75])
        lo, hi = q1 - UMBRAL_TUKEY * (q3 - q1), q3 + UMBRAL_TUKEY * (q3 - q1)
        for i in g.index[(g[var] < lo) | (g[var] > hi)]:
            motivos[i].append(f"{var} {'alto' if anotaciones.at[i, var] > hi else 'bajo'} (Tukey)")
# Mahalanobis robusta: log de cada variable relativa a la mediana de su vértebra, centrado en la mediana y escalado por MAD
rel = np.log(anotaciones[VARS] / anotaciones.groupby("vertebra", observed=True)[VARS].transform("median"))
zr = (rel - rel.median()) / stats.median_abs_deviation(rel, scale="normal")
d2 = np.einsum("ij,jk,ik->i", zr.values, np.linalg.inv(np.cov(zr.T)), zr.values)
for i in anotaciones.index[d2 > stats.chi2.ppf(ALPHA, len(VARS))]:
    motivos[i].append("combinación área/aspecto/relleno (Mahalanobis)")
# Coherencia anatómica: dentro de cada imagen, la posición vertical debe crecer de L1 a L5
anotaciones["orden"] = anotaciones["vertebra"].cat.codes
anotaciones = anotaciones.sort_values(["archivo", "orden"])
prev_cy = anotaciones.groupby("archivo")["cy_rel"].shift(1)
next_cy = anotaciones.groupby("archivo")["cy_rel"].shift(-1)
incoherente = (prev_cy.notna() & (anotaciones["cy_rel"] <= prev_cy)) | (next_cy.notna() & (anotaciones["cy_rel"] >= next_cy))
anotaciones["orden_coherente"] = (~incoherente).astype("int8")
for i in anotaciones.index[incoherente]:
    motivos[i].append("rompe el orden craneocaudal L1→L5")
anotaciones["motivo_atipico"] = [("; ".join(motivos[i]) or None) for i in anotaciones.index]
anotaciones["es_atipico"] = anotaciones["motivo_atipico"].notna().astype("int8")
print(f"Inválidas: {n_fuera + n_degenerados} | atípicas: {anotaciones.es_atipico.sum()} de {len(anotaciones)} | "
      f"rompen el orden L1→L5: {int(incoherente.sum())}")
graficos.atipicos(anotaciones, CLASES, VARS, UMBRAL_TUKEY)

**Lectura.** Sólo **5 de 169** contornos quedan fuera de las vallas, todos **L3** (4 por relleno bajo, 1 por área alta), y ninguno rompe el orden L1→L5: lo que está anotado es confiable. Se conservan marcados para revisión manual.

### 3.3 Descriptores de forma con momentos de OpenCV (P5)

Cada polígono se trata como un contorno (`cv2.moments`): área m₀₀ (validada contra la fórmula de Gauss), centroide e **inclinación** del eje principal θ = ½·atan2(2μ₁₁, μ₂₀ − μ₀₂) con los momentos centrales, además de excentricidad, circularidad, solidez, momentos de Hu e intensidad y contraste dentro del polígono. Por imagen, el **Cobb aproximado** es la diferencia entre la vértebra más y la menos inclinada; por par de vértebras consecutivas (segmento) se mide el desplazamiento horizontal relativo. Escoliosis, normales y código «S» son radiografías **de frente** (la inclinación mide la curva escoliótica); la espondilolistesis es **de perfil** (mide la lordosis) y no se compara con las otras. Con n ≈ 8 por grupo y sin normalidad se usan pruebas **no paramétricas**. La rotación axial (*twist*) no se puede medir en una radiografía 2D.

In [ ]:
intensidad = forma.Intensidad(RAW)                   # intensidad y contraste dentro de cada polígono (máscara OpenCV)
anotaciones = forma.agregar_descriptores(anotaciones, vertices, intensidad)
print(f"Máxima diferencia relativa entre m00 (OpenCV) y el área de Gauss: {(anotaciones.m00 / anotaciones.area_px - 1).abs().max():.1e}")
imagenes["vista"] = np.where(imagenes["diagnostico"] == "Espondilolistesis", "perfil", "frente")

meta = imagenes[["archivo", "diagnostico", "vista"]].astype(str)
cobb = forma.cobb_aproximado(anotaciones, meta)
seg_d = forma.segmentos(anotaciones, CLASES).assign(archivo=lambda d: d["archivo"].astype(str)).merge(meta, on="archivo")
seg_d["desplazamiento_abs"] = seg_d["desplazamiento_rel"].abs()
frente = cobb[cobb.vista == "frente"]
kw = stats.kruskal(*[g.cobb_aprox for _, g in frente.groupby("diagnostico")])
mw = stats.mannwhitneyu(frente.loc[frente.diagnostico == "Escoliosis idiopática (AIS)", "cobb_aprox"],
                        frente.loc[frente.diagnostico == "Normal", "cobb_aprox"], alternative="greater")
lis = seg_d[seg_d.diagnostico == "Espondilolistesis"]
l45 = lis[lis.segmento == "L4-L5"].set_index("archivo")["desplazamiento_abs"]
w = stats.wilcoxon(l45, lis[lis.segmento != "L4-L5"].groupby("archivo")["desplazamiento_abs"].median().loc[l45.index])
print(f"Kruskal-Wallis (frente): p = {kw.pvalue:.2g} | Mann-Whitney escoliosis > normales: p = {mw.pvalue:.2g} | "
      f"Wilcoxon listesis L4–L5 vs. resto: p = {w.pvalue:.2f}")
graficos.alineacion(cobb, seg_d, mw.pvalue, w.pvalue)

**Lectura (P5).** En las radiografías de frente el Cobb aproximado separa la escoliosis de la columna normal (p < 0,001): la inclinación medida con los momentos de cada contorno captura la curva escoliótica. En la espondilolistesis, L4–L5 es el segmento más desplazado, coherente con el nivel que trae el nombre de archivo, pero con n = 8 es una tendencia.

### 3.4 Segunda fuente (NHANES) y orientación

Se aplica a NHANES el mismo control de dominio adaptado a cajas (clases y geometría inválidas, duplicados, orden L1→L5); los atípicos de forma y los momentos no se aplican porque una caja estirada no mide la vértebra. Una primera versión del control de orden marcó 1.709 cajas, pero al revisarlas la mayoría eran **radiografías giradas 90° o invertidas**, no errores de etiqueta. El control se rediseñó en dos pasos, para las dos fuentes: primero la **orientación** de cada imagen (si los centroides se extienden más en vertical o en horizontal y dónde queda L1) y después el **orden**, que exige que L1…L5 sean monótonas a lo largo del eje de la columna. Cada caja se guarda como un polígono de 4 vértices para usar el mismo modelo transaccional.

In [ ]:
img2, anot2, vert2, bit2 = cal.procesar_nhanes(img2, anot2, CLASES)
orient1, _ = cal.orientacion_y_orden(anotaciones.assign(archivo=anotaciones["archivo"].astype(str),
                                                         cx=anotaciones["cx_m"], cy=anotaciones["cy_m"]))
imagenes["orientacion"] = imagenes["archivo"].astype(str).map(orient1.rename(index=str))
completas2 = (anot2.groupby("archivo")["vertebra"].nunique() == len(CLASES)).sum()
rotas = anot2.orden_coherente == 0
print(dict(zip(bit2.paso, bit2.cajas)))
print(f"NHANES: {completas2:,} imágenes completas L1–L5 | {int(rotas.sum())} cajas rompen el orden en "
      f"{anot2.loc[rotas, 'archivo'].nunique()} imágenes | orientación: {img2['orientacion'].value_counts().to_dict()}")
print("Orientación de Lumbar Spine:", imagenes["orientacion"].value_counts().to_dict())

## 4. Base de datos transaccional (SQLite, 3FN)

```
dataset ──────┐
              ├─< imagen ─────┐
diagnostico ──┘               ├─< anotacion ─< vertice
region ─< vertebra ───────────┘
PK compuestas: anotacion (imagen_id, vertebra_id) · vertice (imagen_id, vertebra_id, orden)
```

* **1FN:** el polígono es un grupo repetitivo de largo variable; cada vértice es una fila de `vertice`.
* **2FN:** `anotacion` y `vertice` tienen **claves compuestas** y ningún atributo depende de una parte de la clave: la resolución, la vista y los metadatos dependen sólo de la imagen (están en `imagen`); el código y el orden craneocaudal, sólo de la vértebra (`vertebra`). En `anotacion` quedan las marcas de calidad, que dependen del par.
* **3FN:** sin dependencias transitivas: región, diagnóstico y fuente viven en sus tablas. No se guardan derivados (caja, área, momentos): se recalculan desde los vértices al cargar el almacén.
* Las dos fuentes comparten tablas: `imagen.dataset_id` indica la fuente y `anotacion.geometria` (`poligono` / `caja`) qué métricas tienen sentido. El DDL completo está en `sql/oltp.sql`.

In [ ]:
con = almacen.crear_oltp(OLTP_DB, imagenes, img2, anotaciones, anot2, vertices, vert2, CLASES, DATA_SHA256, NHANES_SHA256)
print("Violaciones de claves foráneas:", con.execute("PRAGMA foreign_key_check").fetchall() or "ninguna")
almacen.esquema(con)

## 5. Almacén de datos (SQLite, copo de nieve con dos tablas de hechos)

```
dim_region ─< dim_vertebra ─┬───────────────────< fact_vertebra_anotada >─┐
                            └─< dim_segmento ───< fact_segmento >─────────┤
dim_dataset ─────┐                                                        │
                 ├─< dim_imagen ──────────────────────────────────────────┘
dim_diagnostico ─┘
```

Dos tablas de hechos que **comparten dimensiones** (constelación), cada una con su **grano** y con PK compuesta por las claves de sus dimensiones: `fact_vertebra_anotada` (una vértebra anotada en una imagen) y `fact_segmento` (un par de vértebras consecutivas en una imagen). Las jerarquías `región → vértebra`, `dataset → imagen` y `diagnóstico → imagen` se normalizan en tablas propias (**copo de nieve**): extender la tesis a la columna cervical o torácica es agregar filas, sin cambiar los hechos. Las **medidas son atómicas** y responden a una pregunta cada una: `area_norm`, `aspecto`, `relleno` y descriptores de forma (P2), `cy_rel`, `alto_norm` y `orden_coherente` (P3), `inclinacion` y las medidas del segmento (P5); las cantidades (P1, P4) son `COUNT`/`SUM` y los promedios y el Cobb se calculan al consultar. No hay dimensión tiempo: las radiografías no tienen fecha. DDL en `sql/dw.sql`.

In [ ]:
dw = almacen.crear_dw(DW_DB, OLTP_DB, CLASES, intensidad)   # dimensiones por SQL (ATTACH); hechos desde los vértices
print("Violaciones de claves foráneas:", dw.execute("PRAGMA foreign_key_check").fetchall() or "ninguna")
almacen.esquema(dw)

In [ ]:
q = lambda sql: pd.read_sql(sql, dw)
print("P1 y P4 — cobertura y representatividad: imágenes, anotadas y completas por fuente y diagnóstico")
display(q('''SELECT ds.nombre AS dataset, dg.nombre AS diagnostico, COUNT(*) AS imagenes, SUM(anotada) AS anotadas,
                    SUM(completa) AS completas, SUM(sexo = 'F') AS mujeres, ROUND(AVG(edad), 1) AS edad_media
             FROM dim_imagen i JOIN dim_dataset ds USING (dataset_key) JOIN dim_diagnostico dg USING (diagnostico_key)
             GROUP BY ds.nombre, dg.nombre ORDER BY imagenes DESC'''))
print("P1, P2 y P3 — balance, calidad y orden por vértebra")
display(q('''SELECT v.codigo AS vertebra, COUNT(*) AS anotaciones, SUM(es_atipico) AS atipicas,
                    SUM(1 - orden_coherente) AS rompen_orden, ROUND(AVG(relleno), 3) AS relleno_medio_poligonos
             FROM fact_vertebra_anotada f JOIN dim_vertebra v USING (vertebra_key)
             GROUP BY v.codigo ORDER BY v.orden_craneocaudal'''))
print("P3 — orientación de las radiografías anotadas")
display(q('''SELECT d.nombre AS dataset, orientacion, COUNT(*) AS imagenes FROM dim_imagen JOIN dim_dataset d USING (dataset_key)
             WHERE anotada = 1 GROUP BY d.nombre, orientacion ORDER BY imagenes DESC'''))
print("P5 — Cobb aproximado (MAX - MIN de la inclinación por imagen, calculado al consultar)")
q('''WITH cobb AS (SELECT imagen_key, MAX(inclinacion) - MIN(inclinacion) AS cobb FROM fact_vertebra_anotada
                WHERE inclinacion IS NOT NULL GROUP BY imagen_key HAVING COUNT(*) >= 4)
     SELECT i.vista, d.nombre AS diagnostico, COUNT(*) AS imagenes, ROUND(AVG(cobb), 1) AS cobb_medio
     FROM cobb JOIN dim_imagen i USING (imagen_key) JOIN dim_diagnostico d USING (diagnostico_key)
     GROUP BY i.vista, d.nombre ORDER BY cobb_medio''')

## 6. Visualización interactiva (Streamlit)

El tablero (`app.py`) lee **sólo del almacén** y empieza por la respuesta: un **veredicto** sobre si los datos alcanzan para entrenar el detector, con el mínimo de imágenes completas como supuesto ajustable desde la barra lateral. Debajo está la evidencia: cobertura imagen × vértebra y balance (P1), atípicos con su motivo (P2), progresión anatómica y orientación (P3), representatividad (P4) y alineación (P5). Los títulos enuncian la conclusión y se recalculan con los filtros; el contexto va en gris y lo que requiere acción, en rojo. En Colab se muestra debajo de la celda mediante el proxy nativo, sin exponer puertos a internet.

In [ ]:
(WORK / "app.py").write_text((BASE / "app.py").read_text(encoding="utf-8"), encoding="utf-8")
if EN_COLAB:
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
                      "--theme.base", "light", "--theme.primaryColor", "#02568A"], cwd=WORK)
    time.sleep(8)
    from google.colab import output
    output.serve_kernel_port_as_iframe(8501, path="/", height=1100)
else:
    print(f"Para abrir el tablero: cd {WORK} && streamlit run app.py")

## 7. Conclusiones

**Respuesta.** *Lumbar Spine* sola **no alcanza** para entrenar el detector L1–L5 (33 imágenes completas, el 16 % de un mínimo supuesto de 200). **Sumando NHANES, los datos públicos sí alcanzan** (4.438 imágenes completas), con dos condiciones que dejó a la vista el control de calidad: **normalizar la orientación** (el 11 % de NHANES llega girada 90° o invertida) y **sumar vistas de frente**, que hoy sólo aporta Lumbar Spine.

* **P1 Cobertura.** El faltante de anotaciones no es aleatorio (escoliosis y código «S»); entre las anotadas la cobertura es casi completa y las clases están balanceadas (razón máxima/mínima 1,01).
* **P2 Calidad.** Normalizar cada vértebra contra sus vecinas evitó los falsos atípicos del encuadre: quedan 5 contornos L3 para revisión manual.
* **P3 Coherencia.** El control de orden es una validación de dominio que ningún método estadístico genérico hace; separar orientación de orden convirtió 1.709 «errores» en 477 imágenes a enderezar y 26 errores reales.
* **P4 Representatividad.** La muestra clínica es chica y dominada por escoliosis adolescente y mujeres; NHANES no trae metadatos.
* **P5 Alineación.** Los momentos de OpenCV estiman un Cobb aproximado que separa escoliosis y normales: los contornos ya aportan el criterio de alineación del SINS.

**Limitaciones.** El código «S» no está documentado por la fuente; no hay identificador de paciente; el mínimo de 200 imágenes es un supuesto (por eso es ajustable); la vista se infiere del diagnóstico y los grupos de P5 tienen n ≈ 8. **Próximos pasos:** enderezar NHANES, entrenar con NHANES y validar con Lumbar Spine, revisar los 5 contornos y buscar una fuente de frente con identificador de paciente.

**Reproducibilidad.** Colab, *Entorno de ejecución → Ejecutar todo*, sin credenciales: los datos crudos se descargan de `data/raw/` del repositorio (SHA-256 verificados) y el código auxiliar se clona del mismo repositorio. Genera `spine_oltp.db`, `spine_dw.db` y el tablero.

**Referencias.** Fisher, C. G. et al. (2010). A novel classification system for spinal instability in neoplastic disease. *Spine*, 35(22). · Knaflic, C. N. (2015). *Storytelling with Data*. Wiley. · Tukey, J. W. (1977). *Exploratory Data Analysis*. Addison-Wesley. · OpenCV, *Image Moments*. · RP Team (2025), *Lumbar Spine Dataset* v1, y main-kybbe, *roboflow_nhanes_l1_to_l5_detection* v1, Roboflow Universe, CC BY 4.0.